# Computational Modelling of Alberta Wildfires
<br> Group 33: 301byte
<br> Group members: Arnav Dhablania, Antarip Kashyap, Eleanor Lam
<br> STAT 301
<br> April 10, 2026

<br> Our group is assigned the dataset- Wildfire from Diverse Data in R packages.

In [ ]:

# devtools::install_github("diverse-data-hub/diversedata")

In [ ]:
library(tidyverse)
suppressPackageStartupMessages({
  library(diversedata)
  library(dplyr)
  library(broom)
  library(ggplot2)
  library(AER)
  library(modelr)
  library(glmnet)
  library(car)
  library(leaps)
  library(rsample)
})

## (2) Introduction

Wildfires have been a critical environmental and public safety concern, particularly in dry provinces such as Alberta where seasonal fire activity has become increasingly severe in recent decades. Wildfires can threaten communities, damage infrastructure, degrade air quality, and disrupt ecosystems. Fire growth is influenced by a combination of factors, including weather conditions (temperature, wind speed, relative humidity), landscape characteristics, and suppression response. Understanding what conditions are associated with larger or faster-growing fires is important for improving wildfire risk assessment, supporting resource allocation, and strengthening preparedness strategies.

However, understanding causes of wildfire growth is challenging because of innate interconnectedness of how factors interact, and observational wildfire data often contains complex relationships, missing values, and nonlinear trends. Using an existing wildfire dataset containing fire size and weather measurements provides an opportunity to explore how strongly weather-related variables are associated with wildfire growth patterns.

This can motivate an investigation into two different avenues: first, whether weather conditions in the dataset can explain wildfire spread rates; and second, <b>which combination of variables from the dataset is the most useful predictor of a wildfire's final burned area? (IS THIS THE QUESTION WE ARE DOING?)</b> 

For the former question, we could use the wildfire dataset to examine whether variables such as temperature, relative humidity, and wind_speed have any associations with the rate at which fire spread rates. Especially considering the natural environment of Alberta and it's weather conditions, understanding these relationships could provide insight into how to better prepare for wildfire fighting based on upcoming weather patterns. 

For the latter question, we could use the wildfire dataset to examine whether variables such as <B>REPLACE WITH VARIABLES WE WILL USE IN FUTRUE</B>. In this report, we will be examining futher into this question. Understanding these relationships could provide insight into <b> some takeaway </b>

## (3) Methods and Results

### (a) Data

We begin by previewing the dataset and its provided description. 

In [ ]:
?wildfire

We begin by previewing the dataset and its provided description. 

In [ ]:
data("wildfire")
head(wildfire)
dim(wildfire)

In [ ]:
variables_table <- tibble(
    name = names(wildfire),
    data_Type = sapply(wildfire, class))

# head(variables_table)

print(variables_table, n = 35)

- The Data set has 35 variables with 26,551 observations.
- The following code cell is a table with all the names, and types of variables

- The dataset was provided under the Open Government Licence - Alberta (according to the dataset help text)  by the Government of Alberta, Canada.
- The dataset has an Open Government Liscense from Alberta. According to the liscensing page associated with this type of license,  the following terms must specifically be adhered to-

*"The Information Provider grants you a worldwide, royalty-free, perpetual, non-exclusive licence to use the Information, including for commercial purposes, subject to the terms below."*

*"You are free to:
Copy, modify, publish, translate, adapt, distribute or otherwise use the Information in any medium, mode or format for any lawful purpose.
You must, where you do any of the above:*

**Acknowledge the source of the Information by including any attribution statement specified by the Information Provider(s) and, where possible, provide a link to this licence.
If the Information Provider does not provide a specific attribution statement, or if you are using Information from several Information Providers and multiple attributions are not practical for your product or application, you must use the following attribution statement:**

                 Contains information licensed under the Open Government Licence – Alberta.

*The terms of this licence are important, and if you fail to comply with any of them, the rights granted to you under this licence, or any similar licence granted by the Information Provider, will end automatically."*


> More important the following exemptions must also be ahdered to-

*"This licence does not grant you any right to use:
Personal Information;
Information or Records that are not accessible under applicable laws;
third party rights the Information Provider is not authorized to license;
the names, crests, logos, or other official symbols of the Information Provider; and
Information subject to other intellectual property rights, including patents, trade-marks and official marks."*

## b) Exploratory Data Analysis (EDA)

We begin our data analysis and visualization by loading, cleaning, and wrangling our data. We copy our data into a separate variable and preview again. Our use of diversedata library helps ensure our code is reproducible, since the data isn't coming from a local copy. 

In [ ]:
wildfire_clean <- wildfire # copy of wildfire
head(wildfire_clean)

After cleaning, we check the percentage of missing values for the variables in the original dataset.

In [ ]:
missing_values <- tibble(
    var = names(wildfire), 
    num_missing = colSums(is.na(wildfire)), 
    prop_na = (num_missing / nrow(wildfire))) %>%
    arrange(desc(prop_na)) |>
    filter(prop_na > 0.20)

head(missing_values)

Based on the table above, we see that ia_arrival_at_fire_dat and fire_fighting_start_date variables have a large proportion of missing data (> 20%). This may suggest that we should be prudent when making any inferences using these variables, since it's generalizability may not be very large. We visualize the data to better see discrepancies. 

In [ ]:
missing_prop <- colMeans(is.na(wildfire_clean))

barplot(
    missing_prop,
    las = 2,  # rotate labels
    col = "skyblue",
    ylab = "Proportion Missing",
    main = "Missing Data by Variable"
)


From the visualization above, we see that action_by and ia_access have a significant proportion of missing data. We look and explore to see if there's any class imbalance in categorical variables in the dataset. We start by filtering the data for the desired variables, and we continue by looking at each variables count, and calculate its proportion. 

In [ ]:
cat_imbalance_var <- wildfire_clean |>
    count(general_cause) |>
    mutate(prop = n/sum(n)) |>
    arrange(desc(prop))

cat_imbalance_var

Lightning appears to be the dominant variable in the dataset at 35%, followed by Recreation and Resident at 22% and 17% respectively. Finally, we visualize the data in a way that will help us explore the data or address the question. To best understand our data, I use a bar plot visualization of the correlation of the numeric varables with the current fire size. 

### EDA Plot

A faceted scatterplot with temperature on the x-axis, relative humidity on the y-axis, point size representing final wildfire size (current_size), point colour representing wind speed, and separate facets for general_cause is used to better understnad the data. This plot is relevant because the investation asks how weather and wildfire characteristics are associated with final wildfire size, and this visualization allows us to examine several important variables at the same time. It is especially useful for seeing whether larger fires tend to occur in hotter, drier, and windier conditions, and whether these patterns differ depending on the cause of the fire.

To improve readability and reduce computation time, the plot is created using a random sample of up to 500 observations. Temperature and relative humidity are then plotted, with point colour to represent wind speed, point size to represent final wildfire size, and faceted by general cause to examine how weather conditions and cause are related to wildfire size.

In [ ]:
set.seed(301)
plot_data <- wildfire %>%
  filter(
    !is.na(temperature),
    !is.na(relative_humidity),
    !is.na(wind_speed),
    !is.na(current_size),
    !is.na(general_cause)
  )

options(repr.plot.width = 12, repr.plot.height = 12)
sample_size <- min(500, nrow(plot_data))

plot_data <- plot_data %>%
  slice_sample(n = sample_size) %>%
  mutate(log_current_size = log1p(current_size)) ## i mutated size to log as it looked odd and exponential without doing this

ggplot(plot_data, aes(x = temperature, y = relative_humidity)) +
  geom_point(
    aes(size = log_current_size, colour = wind_speed),
    alpha = 0.4
  ) + facet_wrap(~ general_cause, ncol = 2) +
  labs(
    title = "Wildfire Size by Weather Conditions and Cause",
    subtitle = "Random sample of 500 wildfires",
    x = "Temperature (°C)",
    y = "Relative Humidity (%)",
    colour = "Wind Speed (km/h)",
    size = "Final estimated area burned by the wildfire (Ha)"
  ) + theme_minimal(base_size = 13) + theme(
    plot.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold", size = 12)
  )

Larger wildfires seem to occur more often under lower relative (under 70%) humidity and moderate to warmer temperatures, especially in the Lightning, Incendiary, and Resident groups. The Lightning panel has by far the most observations, suggesting that this cause is very common in the data, but there is no single simple pattern showing that higher temperature or higher wind speed alone always leads to the largest fires, atleast that is not easily percievable here.

But we do see in most cases, the points are most dense and tend towards a negative correlation, so higher temperature, and lower humidity has the most causes of wildfires.

We continue our EDA by looking into the correlation with wildfire size with various numerical variables. This way, we're able to see which numerical may be most relevant to our response variable, current_size. However, it's also important to note that the correlation only looks at linear relationships: there could be other variables that are highly correlated with current_size that have non-linear relationships, and thus would not be well represented in the visual. Regardless, we start by filtering the data, then building a data frame to accurately plot the bar chart. For visualization purposes, we also flip the x and y axis. 

In [ ]:
set.seed(301)
options(repr.plot.width = 10, repr.plot.height = 10)
theme_set(theme_minimal(base_family = "sans"))

num_data <- wildfire %>%
  select(where(is.numeric)) %>%
  filter(!is.na(current_size)) %>%
  na.omit()

cors <- cor(num_data)[, "current_size"]

cor_df <- data.frame(
  variable = names(cors),
  correlation = as.numeric(cors)) %>%
  filter(variable != "current_size") %>%
  arrange(desc(abs(correlation)))

ggplot(cor_df, 
       aes(x = reorder(variable, correlation), y = correlation)) +
  geom_col(fill = "darkblue", alpha = 0.7) +
  coord_flip() +
  labs(
    title = "Correlation of Each Numeric Variable with Current Wildfire Size",
    x = "Variable",
    y = "Correlation with Current Size"
  ) +
  theme(axis.text.x = element_text(size = 10), axis.text.y = element_text(size = 10), 
     text = element_text(family = "sans")) 


From the above visualization, most of these variables are shown to have a positive association with current_size, with the exception of relative_humidity. The strongest associated variables seem to be first_uc_size, first_ex_size_perimeter, and first_bh_size, with correlations about equal to one. This follows, since they are likely other measurements of fire size. While weather-related variables are mildly correlated, they are significantly less compared to size-based variables. 

## c) Methods: Plan

**Method**

To address the question of interest, we used **Multiple Linear Regression** combined with **Backward Stepwise Selection**. This is appropriate since our response variable (current_size) is continuous and we want to evaluate how multiple explanatory variables together affect the outcome.

Since the goal is to identify the most useful predictors without handpicking, backward selection is the perfect choice, as it starts with a full model including all variables and systematically drops the least useful ones based on **AIC** to find the optimal model. Also, since the data is right-skewed, we applied a log transformation to the response variable to ensure that the model satisfies all assumptions for regression.

**Justification**

We chose to use an additive **Multiple Linear Regression** model with a log-transformed wildfire size response, specifically log(current_size), to address our question of interest. This method is appropriate because the response variable, current_size, is continuous, and the EDA suggests that it is highly right-skewed. Applying a log transformation makes the scale of the response more stable and helps reduce the influence of extremely large fires.

Multiple linear regression is also a suitable choice because our question concerns the joint relationship between wildfire size and several predictors at once, such as temperature, relative humidity, wind speed, year, general cause, and geographic variables like latitude and longitude. This method allows us to estimate how each predictor is associated with wildfire size while holding the others fixed. We chose this method because it is a standard course method, it matches the type of response variable we are studying, and it supports an interpretable analysis of how several variables are associated with wildfire size in the dataset.

For an MLR model, a few assumptions are made, including:
- Linearity: that the relationship between the predictors and the response variable is approximately linear (or can be transformed to be linear. To check this assumption of linearity, fitted values can be plotted against the residuals in order to identify any non-linear patterns. Additionally, transformations of the covariates can change the interpretation of the model.  
- Independence of observations: that observations in the wildfire dataset are independent of each other. To diagnose this, a residual plot can be useful to check for any possible correlations. Violation of this assumption could mean that confidence intervals and calculated p-values are no longer valid. 
- Normal distribution: that errors are approximately normally distributed. This can be mitigated if the sample size is large enough (CLT) and by using bootstrapping. To diagnose, Q-Q plots and histograms of the residuals can be used. 
- Homoscedasticity: errors are roughly constant (consistent variance), and don't form "tube shapes" in their distribution. A fitted vs residuals plot is also useful in detecting heteroscedasticity, and violations of this assumption could again mean confidence intervals and p-values are no longer valid. To address this, we can use variance stabilizing transformations of the response, or to use Weighted Least Squares to see how variance changes with the response. 
- Lack of multicollinearity: predictors should not be severely correlated with one another (usually, in this course, VIF over 5 or 10). Violating this could mean that standard errors of our estimates are inflated, making confidence intervals larger and facilitating fewer opportunities to reject the null hypothesis. As mentioned earlier, VIF can be used to diagnose multicollinearity, which checks the increase in the standard error of the coefficients when fitted alone compared to when fitted with all other variables.

Potential limitations/weaknesses also include
- Fire spread rate/the relatinoship might not be linear - meaning that an MLR model may oversimplify the regression
- Confounding variables may exist, like  fire_type or weather_conditions_over_fire, which may change the interpretations/interpretability of the model coefficients, especially where causality may be related
- Categorical imbalance - since it's an interactive model, will affect how the coefficients are calculated

**Model Implementation**

**MAYBE PUT A DESCRIPTION HERE?** - putting my description below but need to add splitting data (I did 80/20 split)

In this section, I prepare the dataset by filtering out any missing values for our response and predictor variables to make sure that the stepwise model fits our data well. I then applied a log transformation to the response variable and added 1 to address the extreme right-skewness of the fire sizes. I then fit a full multiple linear regression model that contains all potential geographic, environmental, and categorical predictors and predicts the transformed wildfire size accordingly. I applied backward stepwise selection using AIC to drop the least useful variables. In the end, I calculate the VIF on this chosen model to verify that the remaining predictors do not violate multicollinearity.

In [ ]:
model_data <- wildfire %>%
    filter(!is.na(current_size), !is.na(temperature), !is.na(relative_humidity), !is.na(wind_speed), !is.na(latitude), !is.na(longitude), !is.na(year), !is.na(general_cause)) %>%
    mutate(log_current_size = log(current_size + 1), general_cause = factor(general_cause))

set.seed(123)
data_split <- initial_split(model_data, prop = 0.8, strata = log_current_size)
train_data <- training(data_split)
test_data <- testing(data_split)

full_model <- lm(log(current_size + 1) ~ temperature + wind_speed + relative_humidity + fire_spread_rate + latitude + longitude + year + general_cause, data = train_data)
stepwise_model <- step(full_model, direction = "backward", trace = 0)
test_predictions <- predict(stepwise_model, newdata = test_data)
model_vif <- vif(stepwise_model)

test_metrics <- tibble(actual = test_data$log_current_size, predicted = test_predictions) %>% 
    summarise(RMSE = sqrt(mean((actual - predicted)^2)), R_squared = cor(actual, predicted)^2)

print(test_metrics)
summary(stepwise_model)
vif(stepwise_model)

**Table of Results**

To report the results of the multiple linear regression, we have generated a summary table containing the estimated coefficients, 95% confidence intervals, and p-values for each predictor that remained after the backward stepwise selection. This allows us to easily identify the direction, magnitude, and statistical significance of each weather variable's relationship with the log-transformed wildfire size.

In [ ]:
table <- tidy(stepwise_model, conf.int = TRUE) %>%
    mutate(estimate = round(estimate, 4), conf.low = round(conf.low, 4), conf.high = round(conf.high, 4), p.value = round(p.value, 4)) %>%
    select(term, estimate, conf.low, conf.high, p.value) %>%
    arrange(desc(estimate))
table

**Visualization on test data**

We will now plot out our model to see where 

In [ ]:
test_results <- tibble(actual = test_data$log_current_size, predicted = test_predictions)

ggplot(test_results, aes(x = actual, y = predicted)) +
  geom_point(alpha = 0.4) +
  geom_abline(intercept = 0, slope = 1, color = "red", linetype = "dashed", size = 1) +
  labs(title = "Actual vs. Predicted Wildfire Size (Log Scale)",
       x = "Actual Log(Current Size + 1)",
       y = "Predicted Log(Current Size + 1)") +
  theme_minimal()

**STOPPING AT INTERPRETATION - STILL NEED TO DO THAT** 